# Simple Data Loading with Polars

Optimized data loading with proper data types for large datasets

In [2]:
import polars as pl
import pandas as pd
from pathlib import Path

# Set up paths
base_dir = Path("..")
sample_train_path = base_dir / "data" / "sample" / "train_sample.parquet"
sample_test_path = base_dir / "data" / "sample" / "test_sample.parquet"
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

In [3]:
# directly load parquet files into pandas variables (no conversion from Polars)
# sample files
sample_train_pd = pd.read_parquet(sample_train_path)
sample_test_pd  = pd.read_parquet(sample_test_path)

# full files
full_train_pd   = pd.read_parquet(full_train_path)
full_test_pd    = pd.read_parquet(full_test_path)

# now you have four pandas DataFrames ready for inspection




## Load data (fragment or full)

In [ ]:
full_train_pd.head(100).sort_values(by='ts_index', ascending=False)

,id,code,sub_code,sub_category,horizon,ts_index,feature_a,feature_b,feature_c,feature_d,...,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch,y_target,weight
99,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__1__114,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,1,114,28,13.934161,11.460605,8.399934,...,-0.003147,-0.202086,-0.009950,-0.005610,-0.164573,3.276384,0.129198,1,-0.445647,524.042908
98,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__3__114,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,3,114,26,5.446115,14.073915,6.944192,...,-0.003147,-0.202086,-0.009950,-0.005610,-0.164573,3.276384,0.129198,1,-0.912869,395.659068
97,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__114,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,10,114,19,4.358622,9.114108,7.807721,...,-0.003147,-0.202086,-0.009950,-0.005610,-0.164573,3.276384,0.129198,1,-1.813422,225.379465
96,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__114,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,25,114,4,1.109716,4.057972,16.575319,...,-0.003147,-0.202086,-0.009950,-0.005610,-0.164573,3.276384,0.129198,1,-2.602518,140.720544
95,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__113,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,10,113,20,4.864500,4.190726,10.801761,...,-0.002945,-0.189975,-0.009398,-0.005247,-0.045417,3.198393,0.122357,1,-1.849698,214.686928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__90,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,25,90,28,2.303825,7.696209,12.896100,...,-0.001622,-0.103809,-0.005135,NaN,-0.174660,2.738606,0.109204,1,-0.437398,41.948761
2,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__3__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,3,89,51,9.585452,1.076268,9.004147,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.362894,115.953552
1,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__1__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,1,89,53,2.858806,5.050617,15.906651,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.315583,150.075406
3,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,10,89,44,8.840588,15.034634,4.170780,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.667023,64.573073


In [7]:
for name, df in [('sample_train', sample_train_pd), ('sample_test', sample_test_pd),
                 ('full_train', full_train_pd), ('full_test', full_test_pd)]:
    if 'ts_index' in df.columns:
        print(f"{name} ts_index unique: {df['ts_index'].nunique()}")
    else:
        print(f"{name} has no ts_index column")

sample_train ts_index unique: 3587
sample_test ts_index unique: 775
full_train ts_index unique: 3601
full_test ts_index unique: 775


In [8]:
full_train_pd.groupby(['code','sub_code','sub_category'])['ts_index'].apply(lambda x: x.is_monotonic_increasing).all()

True

In [9]:
full_test_pd.groupby(['code','sub_code','sub_category'])['ts_index'].apply(lambda x: x.is_monotonic_increasing).all()   

True

In [12]:
sample_train_pd = sample_train_pd.sort_values(['code','sub_code','sub_category','ts_index'])
sample_test_pd = sample_test_pd.sort_values(['code','sub_code','sub_category','ts_index'])

In [13]:

sample_test_pd.groupby(['code','sub_code','sub_category'])['ts_index'].apply(lambda x: x.is_monotonic_increasing).all()

True

In [14]:
sample_train_pd.groupby(['code','sub_code','sub_category'])['ts_index'].apply(lambda x: x.is_monotonic_increasing).all()

True

In [18]:
print(full_train_pd.head())

                                     id      code  sub_code sub_category  \
0  W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__89  W2MW3G2L  J0G2B0KU     PZ9S1Z4V   
1   W2MW3G2L__J0G2B0KU__PZ9S1Z4V__1__89  W2MW3G2L  J0G2B0KU     PZ9S1Z4V   
2   W2MW3G2L__J0G2B0KU__PZ9S1Z4V__3__89  W2MW3G2L  J0G2B0KU     PZ9S1Z4V   
3  W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__89  W2MW3G2L  J0G2B0KU     PZ9S1Z4V   
4  W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__90  W2MW3G2L  J0G2B0KU     PZ9S1Z4V   

   horizon  ts_index  feature_a  feature_b  feature_c  feature_d  ...  \
0       25        89         29  16.364093   7.464023   5.966933  ...   
1        1        89         53   2.858806   5.050617  15.906651  ...   
2        3        89         51   9.585452   1.076268   9.004147  ...   
3       10        89         44   8.840588  15.034634   4.170780  ...   
4       25        90         28   2.303825   7.696209  12.896100  ...   

   feature_ca  feature_cb  feature_cc  feature_cd  feature_ce  feature_cf  \
0   -0.001686   -0.105328  

In [21]:
print(full_train_pd.tail())

                                            id      code  sub_code  \
5337409  83EG83KQ__R571RU17__PHHHVYZI__3__3522  83EG83KQ  R571RU17   
5337410  83EG83KQ__R571RU17__PHHHVYZI__3__3523  83EG83KQ  R571RU17   
5337411  83EG83KQ__R571RU17__PHHHVYZI__1__3523  83EG83KQ  R571RU17   
5337412  83EG83KQ__R571RU17__PHHHVYZI__1__3524  83EG83KQ  R571RU17   
5337413  83EG83KQ__R571RU17__PHHHVYZI__3__3524  83EG83KQ  R571RU17   

        sub_category  horizon  ts_index  feature_a  feature_b  feature_c  \
5337409     PHHHVYZI        3      3522          2   2.756940   2.611558   
5337410     PHHHVYZI        3      3523          1   3.560270   2.992240   
5337411     PHHHVYZI        1      3523          1   7.978713   8.346381   
5337412     PHHHVYZI        1      3524          0   3.120271  13.186458   
5337413     PHHHVYZI        3      3524          0  16.414658   2.483409   

         feature_d  ...  feature_ca  feature_cb  feature_cc  feature_cd  \
5337409  14.494289  ...   -0.000114   -0.000196

In [19]:
print(full_test_pd.head())

                                       id      code  sub_code sub_category  \
0   W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647  W2MW3G2L  495MGHFJ     PZ9S1Z4V   
1  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647  W2MW3G2L  495MGHFJ     PZ9S1Z4V   
2  W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647  W2MW3G2L  495MGHFJ     PZ9S1Z4V   
3   W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647  W2MW3G2L  495MGHFJ     PZ9S1Z4V   
4  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648  W2MW3G2L  495MGHFJ     PZ9S1Z4V   

   horizon  ts_index  feature_a  feature_b  feature_c  feature_d  ...  \
0        3      3647         95  10.365266   3.209321   8.109339  ...   
1       10      3647         88   2.571477  15.234848  16.505699  ...   
2       25      3647         71   5.524709   6.931663   8.939537  ...   
3        1      3647         97  10.293758  14.893660   9.435544  ...   
4       10      3648         87  14.776194   7.701180   6.228968  ...   

   feature_by  feature_bz  feature_ca  feature_cb  feature_cc  feature_cd  \
0   -0.000832  

In [ ]:
print(full_train_pd.tail())

                                            id      code  sub_code  \
5337409  83EG83KQ__R571RU17__PHHHVYZI__3__3522  83EG83KQ  R571RU17   
5337410  83EG83KQ__R571RU17__PHHHVYZI__3__3523  83EG83KQ  R571RU17   
5337411  83EG83KQ__R571RU17__PHHHVYZI__1__3523  83EG83KQ  R571RU17   
5337412  83EG83KQ__R571RU17__PHHHVYZI__1__3524  83EG83KQ  R571RU17   
5337413  83EG83KQ__R571RU17__PHHHVYZI__3__3524  83EG83KQ  R571RU17   

        sub_category  horizon  ts_index  feature_a  feature_b  feature_c  \
5337409     PHHHVYZI        3      3522          2   2.756940   2.611558   
5337410     PHHHVYZI        3      3523          1   3.560270   2.992240   
5337411     PHHHVYZI        1      3523          1   7.978713   8.346381   
5337412     PHHHVYZI        1      3524          0   3.120271  13.186458   
5337413     PHHHVYZI        3      3524          0  16.414658   2.483409   

         feature_d  ...  feature_ca  feature_cb  feature_cc  feature_cd  \
5337409  14.494289  ...   -0.000114   -0.000196

In [23]:
print("=== TRAIN ts_index ===")
print("Min:", full_train_pd['ts_index'].min())
print("Max:", full_train_pd['ts_index'].max())
print("Unique count:", full_train_pd['ts_index'].nunique())

print("\n=== TEST ts_index ===")
print("Min:", full_test_pd['ts_index'].min())
print("Max:", full_test_pd['ts_index'].max())
print("Unique count:", full_test_pd['ts_index'].nunique())

print("\n=== OVERLAP CHECK ===")
train_ts = set(full_train_pd['ts_index'].unique())
test_ts = set(full_test_pd['ts_index'].unique())
overlap = train_ts.intersection(test_ts)
print(f"Overlap size: {len(overlap)}")
print(f"Test future only: {len(test_ts - train_ts)}")

=== TRAIN ts_index ===
Min: 1
Max: 3601
Unique count: 3601

=== TEST ts_index ===
Min: 3602
Max: 4376
Unique count: 775

=== OVERLAP CHECK ===
Overlap size: 0
Test future only: 775


In [25]:
print(full_train_pd.groupby('sub_category')['y_target'].agg(['mean','std']).round(3))

               mean     std
sub_category               
DPPUO5X2     -0.181  29.266
NQ58FVQM     -0.997  19.833
PHHHVYZI     -1.447  53.455
PZ9S1Z4V     -1.218  28.187
V8BKY1IV      0.526  19.533


In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np

print("Memory-safe baseline...")

# 1. Target encoding
print("1/5 Target encoding...")
sub_cat_means = full_train_pd.groupby('sub_category')['y_target'].mean().to_dict()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(sub_cat_means).fillna(0)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(sub_cat_means).fillna(0)

# 2. SAFE features (sprawdzamy co istnieje)
print("2/5 Safe features...")
all_possible_features = [col for col in full_train_pd.columns if col.startswith('feature_')] + \
                       ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']

# TYLKO kolumny które ISTNIEJĄ w OBU
features = [col for col in all_possible_features if col in full_train_pd.columns and col in full_test_pd.columns]

print(f"Using {len(features)} features")

# 3. Fillna INPLACE
for col in features:
    full_train_pd[col] = full_train_pd[col].fillna(0)
    full_test_pd[col] = full_test_pd[col].fillna(0)

X_train = full_train_pd[features]
y_train = full_train_pd['y_target']
X_test = full_test_pd[features]

# 4. Categories
cat_features = ['code','sub_code','sub_category']
for cat in cat_features:
    if cat in X_train.columns:
        X_train[cat] = X_train[cat].astype('category')
        X_test[cat] = X_test[cat].astype('category')

print("3/5 Training...")
model = lgb.LGBMRegressor(
    n_estimators=50,
    learning_rate=0.2,
    max_depth=3,
    num_leaves=8,
    subsample=0.6,
    colsample_bytree=0.6,
    categorical_feature=cat_features,
    random_state=42,
    verbosity=1
)

model.fit(X_train, y_train)

print("4/5 Predicting...")
preds = model.predict(X_test)

print("5/5 Submission...")
submission = pd.DataFrame({'id': full_test_pd['id'], 'prediction': preds})
submission.to_csv('submission_safe.csv', index=False)
print("✅ submission_safe.csv GOTOWY! (90s total)")
print(f"Shape: {submission.shape}")


In [40]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc
import warnings
warnings.filterwarnings('ignore')

print("Memory-safe baseline...")
print("1/5 Target encoding...")

def prepare_features(full_train_pd, full_test_pd):
    """Convert categorical + target encoding"""
    
    # Target encoding
    cat_means = full_train_pd.groupby('sub_category')['y_target'].mean()
    full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(cat_means)
    full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(cat_means)
    
    # Convert categorical to 'category' dtype
    categorical_cols = ['code', 'sub_code', 'sub_category']
    for col in categorical_cols:
        if col in full_train_pd.columns:
            full_train_pd[col] = full_train_pd[col].astype('category')
            full_test_pd[col] = full_test_pd[col].astype('category')
    
    # Recent features
    full_train_pd['recency'] = 4000 - full_train_pd['ts_index']
    full_train_pd['recent_flag'] = (full_train_pd['ts_index'] > 3000).astype(np.int8)
    full_test_pd['recency'] = 4000 - full_test_pd['ts_index']
    full_test_pd['recent_flag'] = (full_test_pd['ts_index'] > 3000).astype(np.int8)
    
    print("✅ Features prepared")
    return categorical_cols

print("2/5 Safe features...")
categorical_cols = prepare_features(full_train_pd, full_test_pd)

print("3/5 Training...")
sample_weights = np.exp((4000 - full_train_pd['ts_index']) / 500).astype(np.float32)

# FIXED: NO early_stopping → tylko train set
train_data = lgb.Dataset(
    data=full_train_pd.drop(columns=['id', 'y_target']),
    label=full_train_pd['y_target'],
    categorical_feature=categorical_cols,
    weight=sample_weights,
    free_raw_data=True
)

params = {
    'objective': 'regression', 
    'metric': 'mae',
    'n_estimators': 300,  # Fixed iterations
    'learning_rate': 0.1,
    'max_depth': 4,
    'num_leaves': 32,
    'feature_fraction': 0.6,
    'verbose': -1
}

# NO callbacks → NO early stopping error
model = lgb.train(params, train_data)

del train_data, sample_weights
gc.collect()

print("4/5 Predicting...")
predictions = model.predict(full_test_pd.drop(columns=['id']), 
                          num_iteration=model.best_iteration)

print("5/5 Submission...")
submission = pd.DataFrame({
    'id': full_test_pd['id'],
    'prediction': predictions
})

submission.to_csv('submission_safe.csv', index=False)
print("✅ submission_safe.csv GOTOWY!")
print(f"Shape: {submission.shape}")
print(f"Pred mean: {predictions.mean():.3f}")


Memory-safe baseline...
1/5 Target encoding...
2/5 Safe features...
✅ Features prepared
3/5 Training...
4/5 Predicting...


LightGBMError: The number of features in data (94) is not the same as it was in training data (98).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

In [41]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Memory-safe baseline v2...")
print("1/5 Safe features...")

# ★ UŻYWAJ TYLKO ORYGINALNYCH KOLUMN (bez nowych feat)
exclude_cols = ['id', 'y_target']
features = [col for col in full_train_pd.columns if col not in exclude_cols]

# Target encoding BEZPIECZNE
cat_means = full_train_pd.groupby('sub_category')['y_target'].mean()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(cat_means)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(cat_means)

features += ['sub_cat_enc']  # DODAJ do OBU
print(f"Using {len(features)} features")

print("2/5 Categorical...")
categorical_cols = ['code', 'sub_code', 'sub_category']
for cat in categorical_cols:
    full_train_pd[cat] = full_train_pd[cat].astype('category')
    full_test_pd[cat] = full_test_pd[cat].astype('category')

print("3/5 Training...")
# FIXED: .copy() przed modyfikacjami
X_train = full_train_pd[features].copy()
y_train = full_train_pd['y_target'].copy()
X_test = full_test_pd[features].copy()

# Recent weighting
sample_weights = np.exp((4000 - X_train['ts_index']) / 500).astype(np.float32)

train_data = lgb.Dataset(X_train, label=y_train, 
                        categorical_feature=categorical_cols,
                        weight=sample_weights)

params = {
    'objective': 'regression', 
    'metric': 'mae',
    'n_estimators': 500,
    'learning_rate': 0.08,
    'max_depth': 6,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'verbose': -1
}

model = lgb.train(params, train_data)

print("4/5 Predicting...")
predictions = model.predict(X_test)

print("5/5 Submission...")
submission = pd.DataFrame({
    'id': full_test_pd['id'],
    'prediction': predictions
})

submission.to_csv('submission_safe_v2.csv', index=False)
print("✅ submission_safe_v2.csv GOTOWY!")
print(f"Shape: {submission.shape}")
print(f"Pred mean: {predictions.mean():.3f}")


Memory-safe baseline v2...
1/5 Safe features...
Using 99 features
2/5 Categorical...
3/5 Training...


MemoryError: Unable to allocate 3.50 GiB for an array with shape (88, 5337414) and data type float64

In [42]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Baseline v2 - NO copy...")
print("1/5 Features...")

# TYLKO sub_cat_enc (bezpieczne)
cat_means = full_train_pd.groupby('sub_category')['y_target'].mean()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(cat_means)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(cat_means)

# Features BEZ copy
features = [col for col in full_train_pd.columns if col not in ['id', 'y_target']]
print(f"Using {len(features)} features")

print("2/5 Categorical...")
categorical_cols = ['code', 'sub_code', 'sub_category']
for cat in categorical_cols:
    full_train_pd[cat] = full_train_pd[cat].astype('category')
    full_test_pd[cat] = full_test_pd[cat].astype('category')

print("3/5 Training...")
# NO .copy() → LightGBM czyta views
sample_weights = np.exp((4000 - full_train_pd['ts_index']) / 500).astype(np.float32)

train_data = lgb.Dataset(
    full_train_pd[features], 
    label=full_train_pd['y_target'],
    categorical_feature=categorical_cols,
    weight=sample_weights,
    free_raw_data=True
)

params = {
    'objective': 'regression', 
    'metric': 'mae',
    'n_estimators': 400,
    'learning_rate': 0.08,
    'max_depth': 5,
    'num_leaves': 48,
    'feature_fraction': 0.7,
    'verbose': -1
}

model = lgb.train(params, train_data)

print("4/5 Predicting...")
predictions = model.predict(full_test_pd[features])

print("5/5 Submission...")
submission = pd.DataFrame({
    'id': full_test_pd['id'],
    'prediction': predictions
})

submission.to_csv('submission_safe_v2.csv', index=False)
print("✅ submission_safe_v2.csv GOTOWY!")
print(f"Shape: {submission.shape}")


Baseline v2 - NO copy...
1/5 Features...
Using 98 features
2/5 Categorical...
3/5 Training...


MemoryError: Unable to allocate 3.38 GiB for an array with shape (85, 5337414) and data type float64

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Baseline - NO slicing...")
print("1/5 Target encoding...")

# Target encoding BEZPIECZNIE
cat_means = full_train_pd.groupby('sub_category')['y_target'].mean()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(cat_means)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(cat_means)

print("2/5 Categorical...")
categorical_cols = ['code', 'sub_code', 'sub_category']
for cat in categorical_cols:
    full_train_pd[cat] = full_train_pd[cat].astype('category')
    full_test_pd[cat] = full_test_pd[cat].astype('category')

print("3/5 Training...")
# Recent weights
sample_weights = np.exp((4000 - full_train_pd['ts_index']) / 500).astype(np.float32)

# ★ BEZ features list - LightGBM sam wybiera!
train_data = lgb.Dataset(
    full_train_pd.drop(columns=['id', 'y_target']), 
    label=full_train_pd['y_target'],
    categorical_feature=categorical_cols,
    weight=sample_weights,
    free_raw_data=True
)

params = {
    'objective': 'regression', 
    'metric': 'mae',
    'n_estimators': 400,
    'learning_rate': 0.08,
    'max_depth': 5,
    'num_leaves': 48,
    'feature_fraction': 0.7,
    'verbose': -1
}

model = lgb.train(params, train_data)

print("4/5 Predicting...")
# ★ BEZ slicing - LightGBM sam pomija 'id'
predictions = model.predict(full_test_pd)

print("5/5 Submission...")
submission = pd.DataFrame({
    'id': full_test_pd['id'],
    'prediction': predictions
})

submission.to_csv('submission_safe_v2.csv', index=False)
print("✅ submission_safe_v2.csv GOTOWY!")
print(f"Shape: {submission.shape}")



Baseline - NO slicing...
1/5 Target encoding...
2/5 Categorical...
3/5 Training...
4/5 Predicting...


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: id: object

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("FINAL SUBMISSION FOR KAGGLE - POPRAWIONA WERSJA")
print("="*60)

# ============================================
# 1/5 TARGET ENCODING
# ============================================
print("\n1/5 Target encoding...")

# Target encoding dla sub_category
cat_means = full_train_pd.groupby('sub_category')['y_target'].mean()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(cat_means).fillna(0)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(cat_means).fillna(0)

# ============================================
# 2/5 FEATURE SELECTION
# ============================================
print("2/5 Feature selection...")

# Lista wszystkich potencjalnych feature'ów
all_features = [col for col in full_train_pd.columns if col.startswith('feature_')] + \
               ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']

# Wybieramy tylko te, które istnieją w obu zbiorach
features = [col for col in all_features 
            if col in full_train_pd.columns and col in full_test_pd.columns]

print(f"Using {len(features)} features")

# ============================================
# 3/5 DATA PREPARATION - POPRAWIONA!
# ============================================
print("3/5 Data preparation...")

# Najpierw robimy kopie, żeby nie modyfikować oryginałów
X_train = full_train_pd[features].copy()
X_test = full_test_pd[features].copy()
y_train = full_train_pd['y_target'].copy()

# Dla kolumn numerycznych - fillna 0
numeric_features = [col for col in features if col not in ['code', 'sub_code', 'sub_category']]
for col in numeric_features:
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

# Dla kolumn kategorycznych - fillna specjalną wartością 'UNKNOWN'
categorical_cols = ['code', 'sub_code', 'sub_category']
for cat in categorical_cols:
    if cat in X_train.columns:
        # Wypełniamy braki stringiem 'UNKNOWN' (NIE liczbą!)
        X_train[cat] = X_train[cat].fillna('UNKNOWN').astype('category')
        X_test[cat] = X_test[cat].fillna('UNKNOWN').astype('category')

# Wagi czasowe (ważniejsze nowsze obserwacje)
sample_weights = np.exp((4000 - full_train_pd['ts_index']) / 500).astype(np.float32)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Kolumny kategoryczne: {categorical_cols}")

# ============================================
# 4/5 TRAINING
# ============================================
print("4/5 Training LightGBM model...")

model = lgb.LGBMRegressor(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=5,
    num_leaves=48,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    verbosity=-1
)

model.fit(
    X_train, 
    y_train,
    sample_weight=sample_weights,
    categorical_feature=categorical_cols
)

# ============================================
# 5/5 PREDICTION & SUBMISSION
# ============================================
print("5/5 Generating predictions and submission...")

predictions = model.predict(X_test)

submission = pd.DataFrame({
    'id': full_test_pd['id'],
    'prediction': predictions
})

submission.to_csv('submission_kaggle_final.csv', index=False)

print("\n" + "="*60)
print("✅ GOTOWE! Plik: submission_kaggle_final.csv")
print(f"📊 Liczba predykcji: {len(submission)}")
print(f"📋 Próbka (pierwsze 10 wierszy):")
print(submission.head(10))
print("="*60)

# ============================================
# DIAGNOSTYKA
# ============================================
print("\n📈 STATYSTYKI PREDYKCJI:")
print(submission['prediction'].describe())

print("\n🔍 SPRAWDZENIE TYPÓW DANYCH:")
print(f"Kolumny kategoryczne w X_train:")
for cat in categorical_cols:
    if cat in X_train.columns:
        print(f"  {cat}: {X_train[cat].dtype}")
        print(f"    Unikalne wartości: {X_train[cat].unique()[:5]}")

FINAL SUBMISSION FOR KAGGLE - POPRAWIONA WERSJA

1/5 Target encoding...


TypeError: Cannot setitem on a Categorical with a new category (0), set the categories first

In [7]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Memory-safe baseline + TIPS...")
print("1/5 Target encoding...")

# Target encoding (TIP: static z train)
sub_cat_means = full_train_pd.groupby('sub_category')['y_target'].mean().to_dict()
full_train_pd['sub_cat_enc'] = full_train_pd['sub_category'].map(sub_cat_means).fillna(0)
full_test_pd['sub_cat_enc'] = full_test_pd['sub_category'].map(sub_cat_means).fillna(0)

# 2. SAFE features (jak pierwszy kod)
print("2/5 Safe features...")
all_possible_features = [col for col in full_train_pd.columns if col.startswith('feature_')] + \
                       ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']

features = [col for col in all_possible_features if col in full_train_pd.columns and col in full_test_pd.columns]
print(f"Using {len(features)} features")

# Fillna INPLACE (bezpieczne)
for col in features:
    full_train_pd[col] = full_train_pd[col].fillna(0)
    full_test_pd[col] = full_train_pd[col].fillna(0)

print("3/5 TIPS: Categories + Recent weights...")
cat_features = ['code','sub_code','sub_category']
for cat in cat_features:
    if cat in features:
        full_train_pd[cat] = full_train_pd[cat].astype('category')
        full_test_pd[cat] = full_test_pd[cat].astype('category')

# TIP 1: Recent weighting (exponential decay)
ts_index = full_train_pd['ts_index'].values
sample_weights = np.exp((4000 - ts_index) / 300).astype(np.float32)  # float32 = memory safe

print("4/5 Training...")
model = lgb.LGBMRegressor(
    n_estimators=200,        # Więcej iteracji
    learning_rate=0.08,      # Stabilne uczenie
    max_depth=5,             # TIP 3: Low signal/noise
    num_leaves=32,           # Kontrola overfittingu
    subsample=0.8,           # TIP 2: Hierarchia cat
    colsample_bytree=0.8,
    reg_alpha=0.1,           # Regularizacja
    reg_lambda=0.1,
    categorical_feature=cat_features,
    random_state=42,
    verbosity=-1,            # No spam
    n_jobs=-1                # Wszystkie CPU
)

# Fit Z WAHAMI (TIP 1: recent data cięższe)
model.fit(
    full_train_pd[features], 
    full_train_pd['y_target'],
    sample_weight=sample_weights
)

print("5/5 Predicting...")
preds = model.predict(full_test_pd[features])

print("6/5 Submission...")
submission = pd.DataFrame({'id': full_test_pd['id'], 'prediction': preds})
submission.to_csv('submission_tips.csv', index=False)
print("✅ submission_tips.csv GOTOWY!")
print(f"Shape: {submission.shape}")
print(f"Pred mean: {preds.mean():.3f}")


Memory-safe baseline + TIPS...
1/5 Target encoding...


TypeError: Cannot setitem on a Categorical with a new category (0), set the categories first

In [8]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("Memory-safe baseline + TIPS - POPRAWIONY")
print("="*60)

# ============================================
# 1/5 TARGET ENCODING - POPRAWIONE!
# ============================================
print("\n1/5 Target encoding...")

# Najpierw robimy KOPIE danych
train = full_train_pd.copy()
test = full_test_pd.copy()

# Target encoding - używamy oryginalnych kolumn (jeszcze nie kategorycznych)
sub_cat_means = train.groupby('sub_category')['y_target'].mean().to_dict()
train['sub_cat_enc'] = train['sub_category'].map(sub_cat_means)
test['sub_cat_enc'] = test['sub_category'].map(sub_cat_means)

print("sub_cat_enc dodany")

# ============================================
# 2/5 SAFE FEATURES
# ============================================
print("\n2/5 Safe features...")

all_possible_features = [col for col in train.columns if col.startswith('feature_')] + \
                       ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']

features = [col for col in all_possible_features 
            if col in train.columns and col in test.columns]
print(f"Using {len(features)} features")
print(f"Pierwsze 5: {features[:5]}")

# ============================================
# 3/5 DATA PREPARATION - POPRAWIONE!
# ============================================
print("\n3/5 Data preparation...")

# Przygotuj X i y
X_train = train[features].copy()
X_test = test[features].copy()
y_train = train['y_target'].copy()

# Krok 1: Fillna dla KOLUMN NUMERYCZNYCH (wszystkie poza kategorycznymi)
cat_features = ['code', 'sub_code', 'sub_category']
numeric_features = [col for col in features if col not in cat_features + ['sub_cat_enc']]

print(f"Liczba kolumn numerycznych: {len(numeric_features)}")
print(f"Liczba kolumn kategorycznych: {len(cat_features)}")

# Wypełnij numeryczne kolumny zerami
for col in numeric_features:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

# Wypełnij sub_cat_enc (też numeryczne)
X_train['sub_cat_enc'] = pd.to_numeric(X_train['sub_cat_enc'], errors='coerce').fillna(0)
X_test['sub_cat_enc'] = pd.to_numeric(X_test['sub_cat_enc'], errors='coerce').fillna(0)

# Krok 2: Konwersja kolumn kategorycznych (dopiero TERAZ!)
for cat in cat_features:
    if cat in X_train.columns:
        # Najpierw na string, potem fillna 'UNKNOWN', potem category
        X_train[cat] = X_train[cat].astype(str).fillna('UNKNOWN').astype('category')
        X_test[cat] = X_test[cat].astype(str).fillna('UNKNOWN').astype('category')
        print(f"  {cat}: {X_train[cat].dtype}")

# Krok 3: TIP 1 - Recent weighting (exponential decay)
ts_index = train['ts_index'].values
sample_weights = np.exp((4000 - ts_index) / 300).astype(np.float32)
print(f"Wagi: min={sample_weights.min():.2f}, max={sample_weights.max():.2f}")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# ============================================
# 4/5 TRAINING
# ============================================
print("\n4/5 Training...")

model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=5,
    num_leaves=32,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

model.fit(
    X_train, 
    y_train,
    sample_weight=sample_weights,
    categorical_feature=cat_features
)

print("Model wytrenowany!")

# ============================================
# 5/5 PREDICTION & SUBMISSION
# ============================================
print("\n5/5 Predicting...")
preds = model.predict(X_test)

print("6/5 Submission...")
submission = pd.DataFrame({
    'id': test['id'], 
    'prediction': preds
})

submission.to_csv('submission_tips.csv', index=False)

print("\n" + "="*60)
print("✅ submission_tips.csv GOTOWY!")
print(f"📊 Shape: {submission.shape}")
print(f"📈 Predykcje - mean: {preds.mean():.4f}")
print(f"📈 Predykcje - std: {preds.std():.4f}")
print(f"📈 Predykcje - min: {preds.min():.4f}")
print(f"📈 Predykcje - max: {preds.max():.4f}")
print("="*60)

# ============================================
# DIAGNOSTYKA
# ============================================
print("\n🔍 DIAGNOSTYKA:")
print(f"Typy w X_train:")
print(X_train.dtypes.value_counts())
print(f"\nBrakujące wartości w X_train: {X_train.isnull().sum().sum()}")

Memory-safe baseline + TIPS - POPRAWIONY

1/5 Target encoding...
sub_cat_enc dodany

2/5 Safe features...
Using 91 features
Pierwsze 5: ['feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e']

3/5 Data preparation...
Liczba kolumn numerycznych: 87
Liczba kolumn kategorycznych: 3
  code: category
  sub_code: category
  sub_category: category
Wagi: min=3.78, max=615382.94
X_train shape: (5337414, 91)
X_test shape: (1447107, 91)

4/5 Training...
Model wytrenowany!

5/5 Predicting...
6/5 Submission...

✅ submission_tips.csv GOTOWY!
📊 Shape: (1447107, 2)
📈 Predykcje - mean: -2.6850
📈 Predykcje - std: 42.8713
📈 Predykcje - min: -1295.2749
📈 Predykcje - max: 1332.5400

🔍 DIAGNOSTYKA:
Typy w X_train:
float64     85
int32        2
int64        1
category     1
category     1
category     1
Name: count, dtype: int64

Brakujące wartości w X_train: 0


In [9]:
# 1. Wczytaj submission
submission = pd.read_csv('submission_tips.csv')
print(f"✅ Shape OK: {submission.shape}")  # (1447107, 2)

# 2. Statystyki predykcji
print(f"📈 Predykcje:")
print(f"   Mean: {submission['prediction'].mean():.3f}")
print(f"   Std:  {submission['prediction'].std():.3f}")
print(f"   Min:  {submission['prediction'].min():.3f}")
print(f"   Max:  {submission['prediction'].max():.3f}")

# 3. Sprawdź ID
print(f"\n🔍 Przykładowe ID:")
print(submission['id'].head())

# 4. Ocena jakości (heurystyczna)
pred_mean = submission['prediction'].mean()
if abs(pred_mean) < 10 and submission['prediction'].std() > 1:
    print("✅ PREDYKCJE WYGLĄDAJĄ ROZSĄDNIE (LB ~1.0-2.0)")
else:
    print("⚠️  PREDYKCJE ZBYT SKUTECZNE/STAŁE?")


✅ Shape OK: (1447107, 2)
📈 Predykcje:
   Mean: -2.685
   Std:  42.871
   Min:  -1295.275
   Max:  1332.540

🔍 Przykładowe ID:
0     W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647
1    W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647
2    W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647
3     W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647
4    W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648
Name: id, dtype: object
✅ PREDYKCJE WYGLĄDAJĄ ROZSĄDNIE (LB ~1.0-2.0)


In [25]:
# the submission variable already exists from earlier steps and
# was already written with index=False. no need to rewrite it here.
# pandas will always show a numbered index when printing the DataFrame,
# but that index is *not* part of the CSV file when you use index=False.
# we'll open the file directly to demonstrate.

import pandas as pd

# optional: verify header and shape
csv = pd.read_csv("submission.csv")
print(csv.head())
print(csv.shape)

# show first few lines of the raw file to confirm there is no extra column
with open("submission.csv","r",encoding="utf-8") as f:
    for _ in range(6):
        print(f.readline().strip())

                                       id  prediction
0   W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647   -0.076269
1  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647   -0.076269
2  W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647   -0.076269
3   W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647   -0.076269
4  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648   -0.101833
(1447107, 2)
id,prediction
W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.1018329977628744


In [28]:
# reading back the csv to make sure it contains exactly two columns
submission = pd.read_csv("submission.csv")
print(submission.head())
print(submission.shape)

# reminder: the printed DataFrame includes a pandas index column
# that is not stored in the CSV file itself when using index=False.

# if you open the file manually you can see only "id,prediction" header:

with open("submission.csv","r",encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

                                       id  prediction
0   W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647   -0.076269
1  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647   -0.076269
2  W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647   -0.076269
3   W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647   -0.076269
4  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648   -0.101833
(1447107, 2)
id,prediction
W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.0762688512047847
W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.0762688512047847


In [30]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("FINAL VERSION - BEZ DATA LEAKAGE")
print("="*60)

# ============================================
# 1/5 TARGET ENCODING - CAŁKOWICIE NOWE PODEJŚCIE
# ============================================
print("\n1/5 Target encoding...")

# KOPIE danych - ale najpierw konwertujemy kategoryczne na string!
train = full_train_pd.copy()
test = full_test_pd.copy()

# TYMCZASOWO konwertuj kategoryczne na string, żeby móc robić mapowanie
for col in ['sub_category', 'code', 'sub_code']:
    if col in train.columns:
        train[col] = train[col].astype(str)
    if col in test.columns:
        test[col] = test[col].astype(str)

# Teraz target encoding zadziała
sub_cat_means = train.groupby('sub_category')['y_target'].mean().to_dict()
train['sub_cat_enc'] = train['sub_category'].map(sub_cat_means).fillna(0)
test['sub_cat_enc'] = test['sub_category'].map(sub_cat_means).fillna(0)

print("sub_cat_enc dodany")

# ============================================
# 2/5 FEATURE SELECTION
# ============================================
print("\n2/5 Feature selection...")

feature_cols = [col for col in train.columns if col.startswith('feature_')]
all_features = feature_cols + ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']

features = [col for col in all_features 
            if col in train.columns and col in test.columns]
print(f"Using {len(features)} features")
print(f"Pierwsze 5: {features[:5]}")

# ============================================
# 3/5 DATA PREPARATION
# ============================================
print("\n3/5 Data preparation...")

X_train = train[features].copy()
X_test = test[features].copy()
y_train = train['y_target'].copy()

# Wszystkie kolumny poza kategorycznymi są numeryczne
cat_features = ['code', 'sub_code', 'sub_category']
numeric_features = [col for col in features if col not in cat_features]

print(f"Liczba kolumn numerycznych: {len(numeric_features)}")
print(f"Liczba kolumn kategorycznych: {len(cat_features)}")

# Fillna dla numerycznych - używamy 0 (bezpieczne)
for col in numeric_features:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

# Konwersja kategorycznych na typ 'category' (są już stringami)
for cat in cat_features:
    if cat in X_train.columns:
        X_train[cat] = X_train[cat].astype('category')
        X_test[cat] = X_test[cat].astype('category')
        print(f"  {cat}: {X_train[cat].dtype}")

# Wagi czasowe
ts_index = train['ts_index'].values
sample_weights = np.exp((4000 - ts_index) / 300).astype(np.float32)
print(f"Wagi: min={sample_weights.min():.2f}, max={sample_weights.max():.2f}")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# ============================================
# 4/5 TRAINING
# ============================================
print("\n4/5 Training...")

model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=5,
    num_leaves=32,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

model.fit(
    X_train, 
    y_train,
    sample_weight=sample_weights,
    categorical_feature=cat_features
)

print("Model wytrenowany!")

# ============================================
# 5/5 PREDICTION & SUBMISSION
# ============================================
print("\n5/5 Predicting...")
preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'], 
    'prediction': preds
})

submission.to_csv('submission_final.csv', index=False)

print("\n" + "="*60)
print("✅ submission_final.csv GOTOWY!")
print(f"Shape: {submission.shape}")
print(f"Predykcje - mean: {preds.mean():.4f}")
print(f"Predykcje - std: {preds.std():.4f}")
print(f"Predykcje - min: {preds.min():.4f}")
print(f"Predykcje - max: {preds.max():.4f}")
print("="*60)

# ============================================
# DIAGNOSTYKA
# ============================================
print("\n🔍 DIAGNOSTYKA:")
print(f"Typy w X_train:")
print(X_train.dtypes.value_counts())
print(f"\nBrakujące wartości w X_train: {X_train.isnull().sum().sum()}")

FINAL VERSION - BEZ DATA LEAKAGE

1/5 Target encoding...
sub_cat_enc dodany

2/5 Feature selection...
Using 91 features
Pierwsze 5: ['feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e']

3/5 Data preparation...
Liczba kolumn numerycznych: 88
Liczba kolumn kategorycznych: 3
  code: category
  sub_code: category
  sub_category: category
Wagi: min=3.78, max=615382.94
X_train shape: (5337414, 91)
X_test shape: (1447107, 91)

4/5 Training...
Model wytrenowany!

5/5 Predicting...

✅ submission_final.csv GOTOWY!
Shape: (1447107, 2)
Predykcje - mean: -2.6850
Predykcje - std: 42.8713
Predykcje - min: -1295.2749
Predykcje - max: 1332.5400

🔍 DIAGNOSTYKA:
Typy w X_train:
float64     85
int32        2
int64        1
category     1
category     1
category     1
Name: count, dtype: int64

Brakujące wartości w X_train: 0


In [31]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("FINAL VERSION Z WALIDACJĄ")
print("="*60)

# ============================================
# FUNKCJA METRYKI Z KAGGLE
# ============================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# ============================================
# 1/5 TARGET ENCODING
# ============================================
print("\n1/5 Target encoding...")

train = full_train_pd.copy()
test = full_test_pd.copy()

# Konwersja na string dla bezpieczeństwa
for col in ['sub_category', 'code', 'sub_code']:
    if col in train.columns:
        train[col] = train[col].astype(str)
    if col in test.columns:
        test[col] = test[col].astype(str)

sub_cat_means = train.groupby('sub_category')['y_target'].mean().to_dict()
train['sub_cat_enc'] = train['sub_category'].map(sub_cat_means).fillna(0)
test['sub_cat_enc'] = test['sub_category'].map(sub_cat_means).fillna(0)

# ============================================
# 2/5 FEATURE SELECTION
# ============================================
print("2/5 Feature selection...")

feature_cols = [col for col in train.columns if col.startswith('feature_')]
all_features = feature_cols + ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']
features = [col for col in all_features if col in train.columns and col in test.columns]
print(f"Using {len(features)} features")

# ============================================
# 3/5 DATA PREPARATION
# ============================================
print("3/5 Data preparation...")

X = train[features].copy()
y = train['y_target'].copy()
weights = train['weight'].copy()  # Oryginalne wagi z danych!

# Kolumny numeryczne i kategoryczne
cat_features = ['code', 'sub_code', 'sub_category']
numeric_features = [col for col in features if col not in cat_features]

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

for cat in cat_features:
    if cat in X.columns:
        X[cat] = X[cat].astype('category')

# ============================================
# 4/5 WALIDACJA CZASOWA
# ============================================
print("\n4/5 Walidacja czasowa...")

# Sortuj według czasu
X_sorted = X.sort_values('ts_index')
y_sorted = y.loc[X_sorted.index]
weights_sorted = weights.loc[X_sorted.index]

# TimeSeriesSplit - 5 podziałów
tscv = TimeSeriesSplit(n_splits=5)
val_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
    print(f"\n📊 Fold {fold+1}/5")
    
    X_train_fold = X_sorted.iloc[train_idx]
    X_val_fold = X_sorted.iloc[val_idx]
    y_train_fold = y_sorted.iloc[train_idx]
    y_val_fold = y_sorted.iloc[val_idx]
    w_train_fold = weights_sorted.iloc[train_idx]
    w_val_fold = weights_sorted.iloc[val_idx]
    
    # Wagi czasowe dla treningu (opcjonalnie)
    ts_train = X_train_fold['ts_index'].values
    sample_weights_fold = np.exp((4000 - ts_train) / 300).astype(np.float32)
    
    model_fold = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.08,
        max_depth=5,
        num_leaves=32,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
    
    model_fold.fit(
        X_train_fold.drop(columns=['ts_index']), 
        y_train_fold,
        sample_weight=sample_weights_fold,
        categorical_feature=cat_features
    )
    
    # Predykcja na walidacyjnych
    preds_val = model_fold.predict(X_val_fold.drop(columns=['ts_index']))
    
    # Wynik na walidacyjnych (używając ORYGINALNYCH wag z danych!)
    score = weighted_rmse_score(y_val_fold.values, preds_val, w_val_fold.values)
    val_scores.append(score)
    print(f"   Score: {score:.6f}")

print("\n" + "="*60)
print(f"📈 Średni wynik walidacji: {np.mean(val_scores):.6f}")
print(f"📊 Odchylenie std: {np.std(val_scores):.6f}")
print("="*60)

# ============================================
# 5/5 TRENOWANIE NA CAŁOŚCI I SUBMISSION
# ============================================
print("\n5/5 Trenowanie na całości i submission...")

# Wagi czasowe dla całego zbioru
ts_all = X['ts_index'].values
sample_weights_all = np.exp((4000 - ts_all) / 300).astype(np.float32)

final_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=5,
    num_leaves=32,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

final_model.fit(
    X.drop(columns=['ts_index']), 
    y,
    sample_weight=sample_weights_all,
    categorical_feature=cat_features
)

# Przygotuj test
X_test = test[features].copy()
for col in numeric_features:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)
for cat in cat_features:
    if cat in X_test.columns:
        X_test[cat] = X_test[cat].astype('category')

# Predykcja
preds = final_model.predict(X_test.drop(columns=['ts_index']))

submission = pd.DataFrame({
    'id': test['id'], 
    'prediction': preds
})

submission.to_csv('submission_final.csv', index=False)

print("\n✅ submission_final.csv GOTOWY!")
print(f"Shape: {submission.shape}")
print(f"Predykcje - mean: {preds.mean():.4f}")
print(f"Predykcje - std: {preds.std():.4f}")
print(f"Przewidywany score (z walidacji): {np.mean(val_scores):.6f} ± {np.std(val_scores):.6f}")

FINAL VERSION Z WALIDACJĄ

1/5 Target encoding...
2/5 Feature selection...
Using 91 features
3/5 Data preparation...

4/5 Walidacja czasowa...

📊 Fold 1/5
   Score: 0.000000

📊 Fold 2/5
   Score: 0.000000

📊 Fold 3/5
   Score: 0.000000

📊 Fold 4/5
   Score: 0.000000

📊 Fold 5/5
   Score: 0.000000

📈 Średni wynik walidacji: 0.000000
📊 Odchylenie std: 0.000000

5/5 Trenowanie na całości i submission...

✅ submission_final.csv GOTOWY!
Shape: (1447107, 2)
Predykcje - mean: -1.4887
Predykcje - std: 41.7502
Przewidywany score (z walidacji): 0.000000 ± 0.000000


In [32]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("FINAL VERSION Z WALIDACJĄ - POPRAWIONA")
print("="*60)

# ============================================
# FUNKCJA METRYKI Z KAGGLE (POPRAWIONA)
# ============================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    # Zabezpieczenie przed dzieleniem przez 0
    if denom < 1e-15:  # Jeśli denom jest praktycznie 0
        return 1.0  # Maksymalny błąd
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# ============================================
# 1/5 TARGET ENCODING
# ============================================
print("\n1/5 Target encoding...")

train = full_train_pd.copy()
test = full_test_pd.copy()

# Konwersja na string dla bezpieczeństwa
for col in ['sub_category', 'code', 'sub_code']:
    if col in train.columns:
        train[col] = train[col].astype(str)
    if col in test.columns:
        test[col] = test[col].astype(str)

sub_cat_means = train.groupby('sub_category')['y_target'].mean().to_dict()
train['sub_cat_enc'] = train['sub_category'].map(sub_cat_means).fillna(0)
test['sub_cat_enc'] = test['sub_category'].map(sub_cat_means).fillna(0)

# ============================================
# 2/5 FEATURE SELECTION
# ============================================
print("2/5 Feature selection...")

feature_cols = [col for col in train.columns if col.startswith('feature_')]
all_features = feature_cols + ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']
features = [col for col in all_features if col in train.columns and col in test.columns]
print(f"Using {len(features)} features")

# ============================================
# 3/5 DATA PREPARATION
# ============================================
print("3/5 Data preparation...")

X = train[features].copy()
y = train['y_target'].copy()
weights = train['weight'].copy()

# Kolumny numeryczne i kategoryczne
cat_features = ['code', 'sub_code', 'sub_category']
numeric_features = [col for col in features if col not in cat_features]

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

for cat in cat_features:
    if cat in X.columns:
        X[cat] = X[cat].astype('category')

# ============================================
# 4/5 WALIDACJA CZASOWA Z DIAGNOSTYKĄ
# ============================================
print("\n4/5 Walidacja czasowa z diagnostyką...")

# Sortuj według czasu
X_sorted = X.sort_values('ts_index')
y_sorted = y.loc[X_sorted.index]
weights_sorted = weights.loc[X_sorted.index]

# Sprawdź zakres y_target
print(f"\n📊 Statystyki y_target:")
print(f"  Min: {y.min():.4f}")
print(f"  Max: {y.max():.4f}")
print(f"  Mean: {y.mean():.4f}")
print(f"  Std: {y.std():.4f}")
print(f"  Liczba zer: {(y == 0).sum()} ({(y == 0).mean()*100:.2f}%)")

# TimeSeriesSplit - 5 podziałów
tscv = TimeSeriesSplit(n_splits=5)
val_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
    print(f"\n📊 Fold {fold+1}/5")
    
    X_train_fold = X_sorted.iloc[train_idx]
    X_val_fold = X_sorted.iloc[val_idx]
    y_train_fold = y_sorted.iloc[train_idx]
    y_val_fold = y_sorted.iloc[val_idx]
    w_train_fold = weights_sorted.iloc[train_idx]
    w_val_fold = weights_sorted.iloc[val_idx]
    
    print(f"  Rozmiar treningowy: {len(X_train_fold)}")
    print(f"  Rozmiar walidacyjny: {len(X_val_fold)}")
    
    # Wagi czasowe dla treningu
    ts_train = X_train_fold['ts_index'].values
    sample_weights_fold = np.exp((4000 - ts_train) / 300).astype(np.float32)
    
    model_fold = lgb.LGBMRegressor(
        n_estimators=100,  # Mniejsze dla szybszej walidacji
        learning_rate=0.08,
        max_depth=5,
        num_leaves=32,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
    
    model_fold.fit(
        X_train_fold.drop(columns=['ts_index']), 
        y_train_fold,
        sample_weight=sample_weights_fold,
        categorical_feature=cat_features
    )
    
    # Predykcja na walidacyjnych
    preds_val = model_fold.predict(X_val_fold.drop(columns=['ts_index']))
    
    # Sprawdź predykcje
    print(f"  Predykcje - mean: {preds_val.mean():.4f}")
    print(f"  Predykcje - std: {preds_val.std():.4f}")
    
    # Oblicz komponenty metryki
    denom = np.sum(w_val_fold.values * y_val_fold.values ** 2)
    ratio_num = np.sum(w_val_fold.values * (y_val_fold.values - preds_val) ** 2)
    print(f"  Denominator (w*y^2): {denom:.6f}")
    print(f"  Numerator (w*(y-pred)^2): {ratio_num:.6f}")
    
    # Wynik
    score = weighted_rmse_score(y_val_fold.values, preds_val, w_val_fold.values)
    val_scores.append(score)
    print(f"  Score: {score:.6f}")

print("\n" + "="*60)
print(f"📈 Średni wynik walidacji: {np.mean(val_scores):.6f}")
print(f"📊 Odchylenie std: {np.std(val_scores):.6f}")
print("="*60)

# ============================================
# 5/5 TRENOWANIE NA CAŁOŚCI I SUBMISSION
# ============================================
print("\n5/5 Trenowanie na całości i submission...")

# Wagi czasowe dla całego zbioru
ts_all = X['ts_index'].values
sample_weights_all = np.exp((4000 - ts_all) / 300).astype(np.float32)

final_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=5,
    num_leaves=32,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

final_model.fit(
    X.drop(columns=['ts_index']), 
    y,
    sample_weight=sample_weights_all,
    categorical_feature=cat_features
)

# Przygotuj test
X_test = test[features].copy()
for col in numeric_features:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)
for cat in cat_features:
    if cat in X_test.columns:
        X_test[cat] = X_test[cat].astype('category')

# Predykcja
preds = final_model.predict(X_test.drop(columns=['ts_index']))

# Sprawdź czy predykcje mają sens
print(f"\n📊 Statystyki predykcji na test:")
print(f"  Mean: {preds.mean():.4f}")
print(f"  Std: {preds.std():.4f}")
print(f"  Min: {preds.min():.4f}")
print(f"  Max: {preds.max():.4f}")

submission = pd.DataFrame({
    'id': test['id'], 
    'prediction': preds
})

submission.to_csv('submission_final.csv', index=False)

print("\n✅ submission_final.csv GOTOWY!")
print(f"Shape: {submission.shape}")

FINAL VERSION Z WALIDACJĄ - POPRAWIONA

1/5 Target encoding...
2/5 Feature selection...
Using 91 features
3/5 Data preparation...

4/5 Walidacja czasowa z diagnostyką...

📊 Statystyki y_target:
  Min: -2201.8816
  Max: 2314.4112
  Mean: -0.6659
  Std: 32.5276
  Liczba zer: 1628 (0.03%)

📊 Fold 1/5
  Rozmiar treningowy: 889569
  Rozmiar walidacyjny: 889569
  Predykcje - mean: 0.2054
  Predykcje - std: 11.0130
  Denominator (w*y^2): 68761365.546594
  Numerator (w*(y-pred)^2): 301106861373.421509
  Score: 0.000000

📊 Fold 2/5
  Rozmiar treningowy: 1779138
  Rozmiar walidacyjny: 889569
  Predykcje - mean: 1.1963
  Predykcje - std: 16.6156
  Denominator (w*y^2): 65110030.861520
  Numerator (w*(y-pred)^2): 22947692300527.777344
  Score: 0.000000

📊 Fold 3/5
  Rozmiar treningowy: 2668707
  Rozmiar walidacyjny: 889569
  Predykcje - mean: -0.8509
  Predykcje - std: 10.3784
  Denominator (w*y^2): 62499730.179477
  Numerator (w*(y-pred)^2): 227693172999607.500000
  Score: 0.000000

📊 Fold 4/5
  R

segmenty


In [4]:
# ============================================
# CZĘŚĆ 1/3: Przygotowanie danych
# ============================================

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("CZĘŚĆ 1/3: Przygotowanie danych")
print("="*60)

# ============================================
# 1. WCZYTAJ DANE
# ============================================
print("\n📥 Wczytywanie danych...")

# Masz już full_train_pd i full_test_pd w pamięci?
# Jeśli nie, wczytaj:
# full_train_pd = pd.read_parquet('train.parquet')
# full_test_pd = pd.read_parquet('test.parquet')

train = full_train_pd.copy()
test = full_test_pd.copy()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# ============================================
# 2. TARGET ENCODING
# ============================================
print("\n🎯 Target encoding...")

# Konwersja na string dla bezpieczeństwa
for col in ['sub_category', 'code', 'sub_code']:
    if col in train.columns:
        train[col] = train[col].astype(str)
    if col in test.columns:
        test[col] = test[col].astype(str)

sub_cat_means = train.groupby('sub_category')['y_target'].mean().to_dict()
train['sub_cat_enc'] = train['sub_category'].map(sub_cat_means).fillna(0)
test['sub_cat_enc'] = test['sub_category'].map(sub_cat_means).fillna(0)

print("sub_cat_enc dodany")

# ============================================
# 3. FEATURE SELECTION
# ============================================
print("\n🔍 Feature selection...")

feature_cols = [col for col in train.columns if col.startswith('feature_')]
all_features = feature_cols + ['sub_cat_enc', 'ts_index', 'code', 'sub_code', 'sub_category']
features = [col for col in all_features if col in train.columns and col in test.columns]
print(f"Using {len(features)} features")
print(f"Pierwsze 5: {features[:5]}")

# ============================================
# 4. PRZYGOTOWANIE DANYCH
# ============================================
print("\n🛠️ Data preparation...")

X = train[features].copy()
y = train['y_target'].copy()

# Kolumny numeryczne i kategoryczne
cat_features = ['code', 'sub_code', 'sub_category']
numeric_features = [col for col in features if col not in cat_features]

print(f"Liczba kolumn numerycznych: {len(numeric_features)}")
print(f"Liczba kolumn kategorycznych: {len(cat_features)}")

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

for cat in cat_features:
    if cat in X.columns:
        X[cat] = X[cat].astype('category')

# Przygotuj test
X_test = test[features].copy()
for col in numeric_features:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)
for cat in cat_features:
    if cat in X_test.columns:
        X_test[cat] = X_test[cat].astype('category')

# ============================================
# 5. ZAPIS DO PLIKÓW
# ============================================
print("\n💾 Zapisuję przygotowane dane...")

X.to_pickle('X_train.pkl')
y.to_pickle('y_train.pkl')
X_test.to_pickle('X_test.pkl')
test[['id']].to_pickle('test_ids.pkl')

# Zapisz listę features
with open('features_list.pkl', 'wb') as f:
    pickle.dump({
        'features': features,
        'cat_features': cat_features,
        'numeric_features': numeric_features
    }, f)

print("\n✅ CZĘŚĆ 1 ZAKOŃCZONA!")
print("Zapisano pliki:")
print("  - X_train.pkl")
print("  - y_train.pkl")
print("  - X_test.pkl")
print("  - test_ids.pkl")
print("  - features_list.pkl")
print("="*60)

CZĘŚĆ 1/3: Przygotowanie danych

📥 Wczytywanie danych...
Train shape: (5337414, 94)
Test shape: (1447107, 92)

🎯 Target encoding...
sub_cat_enc dodany

🔍 Feature selection...
Using 91 features
Pierwsze 5: ['feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e']

🛠️ Data preparation...
Liczba kolumn numerycznych: 88
Liczba kolumn kategorycznych: 3

💾 Zapisuję przygotowane dane...

✅ CZĘŚĆ 1 ZAKOŃCZONA!
Zapisano pliki:
  - X_train.pkl
  - y_train.pkl
  - X_test.pkl
  - test_ids.pkl
  - features_list.pkl


In [5]:
# ============================================
# CZĘŚĆ 2/3: Walidacja czasowa
# ============================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("CZĘŚĆ 2/3: Walidacja czasowa")
print("="*60)

# ============================================
# 1. FUNKCJA METRYKI
# ============================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom < 1e-15:
        return 1.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# ============================================
# 2. WCZYTAJ DANE
# ============================================
print("\n📂 Wczytywanie danych...")

X = pd.read_pickle('X_train.pkl')
y = pd.read_pickle('y_train.pkl')

with open('features_list.pkl', 'rb') as f:
    saved = pickle.load(f)
    cat_features = saved['cat_features']

print(f"X shape: {X.shape}")
print(f"cat_features: {cat_features}")

# ============================================
# 3. SZYBKI TEST PARAMETRÓW
# ============================================
print("\n🔍 Szybki test parametrów...")

# Sortuj według czasu
X_sorted = X.sort_values('ts_index')
y_sorted = y.loc[X_sorted.index]

# TimeSeriesSplit - tylko 2 foldy dla szybkiego testu
tscv = TimeSeriesSplit(n_splits=2)

# Testuj różne parametry
param_grid = [
    {'n_estimators': 100, 'learning_rate': 0.08, 'max_depth': 5, 'num_leaves': 32},
    {'n_estimators': 150, 'learning_rate': 0.06, 'max_depth': 6, 'num_leaves': 40},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 7, 'num_leaves': 48},
]

best_score = float('inf')
best_params = None

for params_idx, params in enumerate(param_grid):
    print(f"\n📈 Test {params_idx+1}/{len(param_grid)}: {params}")
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
        X_train_fold = X_sorted.iloc[train_idx]
        X_val_fold = X_sorted.iloc[val_idx]
        y_train_fold = y_sorted.iloc[train_idx]
        y_val_fold = y_sorted.iloc[val_idx]
        
        # Wagi czasowe
        ts_train = X_train_fold['ts_index'].values
        ts_val = X_val_fold['ts_index'].values
        sample_weights = np.exp((4000 - ts_train) / 300).astype(np.float32)
        val_weights = np.exp((4000 - ts_val) / 300).astype(np.float32)
        
        model = lgb.LGBMRegressor(
            **params,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            verbosity=-1,
            n_jobs=-1
        )
        
        model.fit(
            X_train_fold.drop(columns=['ts_index']), 
            y_train_fold,
            sample_weight=sample_weights,
            categorical_feature=cat_features
        )
        
        preds = model.predict(X_val_fold.drop(columns=['ts_index']))
        score = weighted_rmse_score(y_val_fold.values, preds, val_weights)
        fold_scores.append(score)
        print(f"  Fold {fold+1}: {score:.6f}")
    
    mean_score = np.mean(fold_scores)
    print(f"  Średni score: {mean_score:.6f}")
    
    if mean_score < best_score:
        best_score = mean_score
        best_params = params
        print(f"  ⭐ NOWY NAJLEPSZY!")

print("\n" + "="*60)
print(f"🏆 NAJLEPSZE PARAMETRY:")
print(f"   {best_params}")
print(f"📈 Wynik: {best_score:.6f}")
print("="*60)

# ============================================
# 4. PEŁNA WALIDACJA (opcjonalnie)
# ============================================
print("\n⏳ Pełna walidacja (5 foldów) - to potrwa dłużej...")

full_tscv = TimeSeriesSplit(n_splits=5)
full_scores = []

for fold, (train_idx, val_idx) in enumerate(full_tscv.split(X_sorted)):
    print(f"  Fold {fold+1}/5...", end=" ")
    
    X_train_fold = X_sorted.iloc[train_idx]
    X_val_fold = X_sorted.iloc[val_idx]
    y_train_fold = y_sorted.iloc[train_idx]
    y_val_fold = y_sorted.iloc[val_idx]
    
    ts_train = X_train_fold['ts_index'].values
    ts_val = X_val_fold['ts_index'].values
    sample_weights = np.exp((4000 - ts_train) / 300).astype(np.float32)
    val_weights = np.exp((4000 - ts_val) / 300).astype(np.float32)
    
    model = lgb.LGBMRegressor(
        **best_params,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
    
    model.fit(
        X_train_fold.drop(columns=['ts_index']), 
        y_train_fold,
        sample_weight=sample_weights,
        categorical_feature=cat_features
    )
    
    preds = model.predict(X_val_fold.drop(columns=['ts_index']))
    score = weighted_rmse_score(y_val_fold.values, preds, val_weights)
    full_scores.append(score)
    print(f"score={score:.6f}")

print(f"\n📈 Średni wynik (5 foldów): {np.mean(full_scores):.6f} ± {np.std(full_scores):.6f}")

# ============================================
# 5. ZAPISZ WYNIKI
# ============================================
with open('validation_results.pkl', 'wb') as f:
    pickle.dump({
        'best_params': best_params,
        'best_score': best_score,
        'full_scores': full_scores
    }, f)

print("\n✅ CZĘŚĆ 2 ZAKOŃCZONA!")
print("Zapisano validation_results.pkl")

CZĘŚĆ 2/3: Walidacja czasowa

📂 Wczytywanie danych...
X shape: (5337414, 91)
cat_features: ['code', 'sub_code', 'sub_category']

🔍 Szybki test parametrów...

📈 Test 1/3: {'n_estimators': 100, 'learning_rate': 0.08, 'max_depth': 5, 'num_leaves': 32}
  Fold 1: 0.000000
  Fold 2: 0.000000
  Średni score: 0.000000
  ⭐ NOWY NAJLEPSZY!

📈 Test 2/3: {'n_estimators': 150, 'learning_rate': 0.06, 'max_depth': 6, 'num_leaves': 40}
  Fold 1: 0.000000
  Fold 2: 0.000000
  Średni score: 0.000000

📈 Test 3/3: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 7, 'num_leaves': 48}
  Fold 1: 0.000000
  Fold 2: 0.000000
  Średni score: 0.000000

🏆 NAJLEPSZE PARAMETRY:
   {'n_estimators': 100, 'learning_rate': 0.08, 'max_depth': 5, 'num_leaves': 32}
📈 Wynik: 0.000000

⏳ Pełna walidacja (5 foldów) - to potrwa dłużej...
  Fold 1/5... score=0.000000
  Fold 2/5... score=0.000000
  Fold 3/5... score=0.000000
  Fold 4/5... score=0.000000
  Fold 5/5... score=0.000000

📈 Średni wynik (5 foldów): 0.000000 

In [6]:
# ============================================
# CZĘŚĆ 3/3: Trenowanie końcowe i submission
# ============================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("CZĘŚĆ 3/3: Trenowanie końcowe i submission")
print("="*60)

# ============================================
# 1. WCZYTAJ DANE I WYNIKI WALIDACJI
# ============================================
print("\n📂 Wczytywanie danych...")

X = pd.read_pickle('X_train.pkl')
y = pd.read_pickle('y_train.pkl')
X_test = pd.read_pickle('X_test.pkl')
test_ids = pd.read_pickle('test_ids.pkl')

with open('features_list.pkl', 'rb') as f:
    saved = pickle.load(f)
    cat_features = saved['cat_features']

with open('validation_results.pkl', 'rb') as f:
    val_results = pickle.load(f)
    best_params = val_results['best_params']
    val_score = val_results['best_score']

print(f"X_train shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Najlepsze parametry: {best_params}")
print(f"Wynik walidacji: {val_score:.6f}")

# ============================================
# 2. TRENOWANIE KOŃCOWE
# ============================================
print("\n🚀 Trenowanie modelu końcowego...")

# Wagi czasowe dla całego zbioru
ts_all = X['ts_index'].values
sample_weights = np.exp((4000 - ts_all) / 300).astype(np.float32)

print(f"Średnia waga: {sample_weights.mean():.2f}")
print(f"Zakres wag: {sample_weights.min():.2f} - {sample_weights.max():.2f}")

final_model = lgb.LGBMRegressor(
    **best_params,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

# Mierz czas trenowania
import time
start_time = time.time()

final_model.fit(
    X.drop(columns=['ts_index']), 
    y,
    sample_weight=sample_weights,
    categorical_feature=cat_features
)

training_time = time.time() - start_time
print(f"Trenowanie zajęło: {training_time:.2f} sekund")

# ============================================
# 3. FEATURE IMPORTANCE
# ============================================
print("\n📊 Feature importance (top 10):")

feature_importance = pd.DataFrame({
    'feature': X.drop(columns=['ts_index']).columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

print(feature_importance)

# ============================================
# 4. PREDYKCJA
# ============================================
print("\n🔮 Generowanie predykcji...")

preds = final_model.predict(X_test.drop(columns=['ts_index']))

print(f"Statystyki predykcji:")
print(f"  Mean: {preds.mean():.4f}")
print(f"  Std: {preds.std():.4f}")
print(f"  Min: {preds.min():.4f}")
print(f"  Max: {preds.max():.4f}")
print(f"  Q1: {np.percentile(preds, 25):.4f}")
print(f"  Median: {np.median(preds):.4f}")
print(f"  Q3: {np.percentile(preds, 75):.4f}")

# ============================================
# 5. SUBMISSION
# ============================================
print("\n📝 Tworzenie pliku submission...")

submission = pd.DataFrame({
    'id': test_ids['id'], 
    'prediction': preds
})

# Dodaj timestamp do nazwy pliku
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f'submission_{timestamp}.csv'

submission.to_csv(filename, index=False)

print("\n" + "="*60)
print(f"✅ PLIK GOTOWY: {filename}")
print(f"Shape: {submission.shape}")
print(f"Przewidywany score (z walidacji): {val_score:.6f}")
print("="*60)

# ============================================
# 6. PODGLĄD
# ============================================
print("\n📋 Pierwsze 10 wierszy submission:")
print(submission.head(10))

# ============================================
# 7. SPRAWDZENIE FORMATU
# ============================================
print("\n✅ Sprawdzenie formatu:")
print(f"Kolumny: {submission.columns.tolist()}")
print(f"Typy danych:\n{submission.dtypes}")
print(f"Brakujące wartości: {submission.isnull().sum().sum()}")

CZĘŚĆ 3/3: Trenowanie końcowe i submission

📂 Wczytywanie danych...
X_train shape: (5337414, 91)
X_test shape: (1447107, 91)
Najlepsze parametry: {'n_estimators': 100, 'learning_rate': 0.08, 'max_depth': 5, 'num_leaves': 32}
Wynik walidacji: 0.000000

🚀 Trenowanie modelu końcowego...
Średnia waga: 35912.73
Zakres wag: 3.78 - 615382.94
Trenowanie zajęło: 52.27 sekund

📊 Feature importance (top 10):
         feature  importance
0      feature_a         398
37    feature_al         291
88      sub_code         203
89  sub_category         192
86   sub_cat_enc         186
70    feature_bs          58
12     feature_m          56
25     feature_z          54
76    feature_by          52
66    feature_bo          48

🔮 Generowanie predykcji...
Statystyki predykcji:
  Mean: -0.8399
  Std: 33.1218
  Min: -924.8646
  Max: 1489.6275
  Q1: -0.1597
  Median: -0.0072
  Q3: 0.7182

📝 Tworzenie pliku submission...

✅ PLIK GOTOWY: submission_20260308_093511.csv
Shape: (1447107, 2)
Przewidywany score (

In [7]:
# ============================================
# BASELINE: Regresja liniowa na małej próbce
# ============================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("BASELINE - Regresja liniowa")
print("="*60)

# ============================================
# 1. Funkcja metryki Kaggle
# ============================================
def _clip01(x):
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_true, y_pred, w):
    denom = np.sum(w * y_true ** 2)
    if denom < 1e-15:
        return 1.0
    ratio = np.sum(w * (y_true - y_pred) ** 2) / denom
    return float(np.sqrt(1.0 - _clip01(ratio)))

# ============================================
# 2. Wczytaj dane (używając Twoich plików)
# ============================================
print("\n📂 Wczytywanie danych...")

X_train = pd.read_pickle('X_train.pkl')
y_train = pd.read_pickle('y_train.pkl')

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# ============================================
# 3. Weź MAŁĄ próbkę (szybkie testy)
# ============================================
print("\n📊 Próbkowanie danych...")

# Opcja A: Losowa próbka 50k
sample_size = 50000
np.random.seed(42)
idx = np.random.choice(len(X_train), sample_size, replace=False)

X_sample = X_train.iloc[idx]
y_sample = y_train.iloc[idx]

print(f"Próbka: {X_sample.shape}")

# ============================================
# 4. Wybierz tylko numeryczne kolumny
# ============================================
print("\n🔍 Przygotowanie feature'ów...")

# Wszystkie kolumny poza kategorycznymi
cat_features = ['code', 'sub_code', 'sub_category']
numeric_cols = [col for col in X_sample.columns if col not in cat_features + ['ts_index']]

print(f"Używam {len(numeric_cols)} numerycznych feature'ów")

# Proste fillna 0 (ale Ty możesz to zmienić!)
X_numeric = X_sample[numeric_cols].fillna(0).values
y_values = y_sample.values

# ============================================
# 5. Trenuj model
# ============================================
print("\n🚀 Trenowanie regresji liniowej...")

model = LinearRegression()
model.fit(X_numeric, y_values)

# ============================================
# 6. Predykcja i metryka
# ============================================
print("\n📈 Obliczanie metryki...")

y_pred = model.predict(X_numeric)

# Użyj prostych wag (1) do testu
simple_weights = np.ones(len(y_values))
score = weighted_rmse_score(y_values, y_pred, simple_weights)

print(f"Wynik na próbce treningowej: {score:.6f}")

# ============================================
# 7. Sprawdź czy cokolwiek działa
# ============================================
print(f"\n📊 Statystyki:")
print(f"  y_true - mean: {y_values.mean():.4f}, std: {y_values.std():.4f}")
print(f"  y_pred - mean: {y_pred.mean():.4f}, std: {y_pred.std():.4f}")
print(f"  Korelacja: {np.corrcoef(y_values, y_pred)[0,1]:.4f}")

# ============================================
# 8. Szybki test na walidacji czasowej
# ============================================
print("\n⏱️ Szybki test czasowy...")

# Sortuj po czasie
X_sorted = X_train.sort_values('ts_index')
y_sorted = y_train.loc[X_sorted.index]

# Weź pierwsze 80% jako train, ostatnie 20% jako val
split = int(len(X_sorted) * 0.8)
X_train_time = X_sorted.iloc[:split][numeric_cols].fillna(0).values
y_train_time = y_sorted.iloc[:split].values
X_val_time = X_sorted.iloc[split:][numeric_cols].fillna(0).values
y_val_time = y_sorted.iloc[split:].values

# Wagi czasowe (ważniejsze nowsze)
ts_val = X_sorted.iloc[split:]['ts_index'].values
val_weights = np.exp((4000 - ts_val) / 300)

# Trenuj na czasowym train
model_time = LinearRegression()
model_time.fit(X_train_time, y_train_time)

# Predykcja na val
y_pred_time = model_time.predict(X_val_time)

# Metryka
score_time = weighted_rmse_score(y_val_time, y_pred_time, val_weights)
print(f"Wynik na walidacji czasowej: {score_time:.6f}")

print("\n" + "="*60)
print("✅ BASELINE GOTOWY")
print("="*60)

BASELINE - Regresja liniowa

📂 Wczytywanie danych...
X_train shape: (5337414, 91)
y_train shape: (5337414,)

📊 Próbkowanie danych...
Próbka: (50000, 91)

🔍 Przygotowanie feature'ów...
Używam 87 numerycznych feature'ów

🚀 Trenowanie regresji liniowej...

📈 Obliczanie metryki...
Wynik na próbce treningowej: 0.139201

📊 Statystyki:
  y_true - mean: -0.7906, std: 32.4502
  y_pred - mean: -0.7906, std: 4.4487
  Korelacja: 0.1371

⏱️ Szybki test czasowy...
Wynik na walidacji czasowej: 0.000000

✅ BASELINE GOTOWY


In [8]:
import pandas as pd
import numpy as np

print("🔍 SPRAWDZENIE MOŻLIWOŚCI KONWERSJI DO FLOAT16")
print("="*60)

# Wczytaj dane
X_train = pd.read_pickle('X_train.pkl')

# Float16 zakres: -65504 do 65504
FLOAT16_MIN = -65504
FLOAT16_MAX = 65504

# Sprawdź każdą kolumnę numeryczną
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

print(f"\n📊 Liczba kolumn numerycznych: {len(numeric_cols)}")

# Statystyki dla każdej kolumny
stats = []
for col in numeric_cols:
    col_min = X_train[col].min()
    col_max = X_train[col].max()
    col_mean = X_train[col].mean()
    col_std = X_train[col].std()
    
    is_safe = (col_min > FLOAT16_MIN) and (col_max < FLOAT16_MAX)
    
    stats.append({
        'kolumna': col,
        'min': col_min,
        'max': col_max,
        'mean': col_mean,
        'std': col_std,
        'safe_for_float16': is_safe,
        'zakres': col_max - col_min
    })

df_stats = pd.DataFrame(stats)

# Podsumowanie
safe_count = df_stats['safe_for_float16'].sum()
unsafe_count = len(df_stats) - safe_count

print(f"\n✅ Bezpieczne dla float16: {safe_count} kolumn ({safe_count/len(df_stats)*100:.1f}%)")
print(f"❌ NIEBEZPIECZNE dla float16: {unsafe_count} kolumn ({unsafe_count/len(df_stats)*100:.1f}%)")

if unsafe_count > 0:
    print("\n🔴 KOLUMNY Z ZAKRESEM POZA FLOAT16:")
    unsafe_cols = df_stats[~df_stats['safe_for_float16']].sort_values('zakres', ascending=False)
    for idx, row in unsafe_cols.iterrows():
        print(f"  {row['kolumna']}: min={row['min']:.0f}, max={row['max']:.0f}, zakres={row['zakres']:.0f}")

# Sprawdź czy to ekstrema czy cały zakres
print("\n📈 ANALIZA EKSTREMÓW:")
for col in unsafe_cols.head(5)['kolumna']:
    data = X_train[col]
    percentiles = [0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]
    print(f"\n{col}:")
    for p in percentiles:
        print(f"  {p*100:.0f}%: {data.quantile(p):.2f}")

🔍 SPRAWDZENIE MOŻLIWOŚCI KONWERSJI DO FLOAT16

📊 Liczba kolumn numerycznych: 88

✅ Bezpieczne dla float16: 72 kolumn (81.8%)
❌ NIEBEZPIECZNE dla float16: 16 kolumn (18.2%)

🔴 KOLUMNY Z ZAKRESEM POZA FLOAT16:
  feature_ba: min=0, max=31754479, zakres=31754479
  feature_bd: min=0, max=26449709, zakres=26449709
  feature_be: min=0, max=10795225, zakres=10795225
  feature_bf: min=0, max=10374965, zakres=10374965
  feature_bh: min=0, max=9412712, zakres=9412712
  feature_bc: min=0, max=5542532, zakres=5542532
  feature_bb: min=0, max=3630534, zakres=3630534
  feature_bk: min=0, max=3569165, zakres=3569165
  feature_bj: min=0, max=1249108, zakres=1249108
  feature_au: min=0, max=994833, zakres=994833
  feature_aw: min=0, max=787294, zakres=787294
  feature_ax: min=0, max=776179, zakres=776179
  feature_av: min=0, max=618352, zakres=618352
  feature_ay: min=0, max=600895, zakres=600895
  feature_at: min=0, max=550471, zakres=550471
  feature_bi: min=0, max=66063, zakres=66063

📈 ANALIZA EKSTR